<a href="https://colab.research.google.com/github/cwyforjupas-del/tiktok-stuff/blob/cwyforjupas-del-patch-1/tiktok_stuff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

## Setup & Configuration
This notebook scrapes MUET product data from Bauhaus Hong Kong and generates TikTok Seller Center batch upload files.

**Before running:**
1. Ensure the Excel template `Tiktoksellercenter_batchupload_20260617_template.xlsx` is present locally.
2. Run cells in order: Scraper → Integration → Template Filler.
3. By default, the integration cell preserves previously uploaded products (no duplicates). To clear tracking and reprocess all products, set `CLEAR_UPLOAD_TRACKER = True` in that cell.

In [1]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font
import json
import os
from datetime import datetime
import hashlib

# ================= CONFIG =================
TEMPLATE_PATH = "Tiktoksellercenter_batchupload_20260617_template.xlsx"
OUTPUT_DIR = "output"
BRAND_NAME = "MUET"
BRAND_ID = "7618173140417627905"
CURRENCY = "SGD"
BATCH_SIZE = 50

os.makedirs(OUTPUT_DIR, exist_ok=True)

TRACK_FILE = "uploaded_products.json"
if os.path.exists(TRACK_FILE):
    with open(TRACK_FILE, 'r') as f:
        uploaded = json.load(f)
    print(f"Loaded {len(uploaded)} previously uploaded products from {TRACK_FILE}.")
else:
    uploaded = []
    print("No previous upload tracking found. Starting fresh.")

def save_uploaded():
    """Persist the uploaded product list to JSON file."""
    with open(TRACK_FILE, 'w') as f:
        json.dump(uploaded, f, indent=2)
    print(f"Saved {len(uploaded)} tracked uploads to {TRACK_FILE}.")

def generate_stable_sku(product_name, product_url):
    """
    Generate a stable SKU based on product name and URL hash.
    This ensures consistent SKUs across multiple runs (unlike datetime-based SKUs).
    Format: MUET-{date}-{hash_prefix}
    """
    url_hash = hashlib.md5(f"{product_name}_{product_url}".encode()).hexdigest()[:6].upper()
    date_str = datetime.now().strftime('%Y%m%d')
    return f"MUET-{date_str}-{url_hash}"

def fill_template(selected_products, products_db_to_use):
    """
    Fill the TikTok template with selected product data.
    Returns the output file path on success, None on error.
    """
    if not os.path.exists(TEMPLATE_PATH):
        print(f"Error: Template file '{TEMPLATE_PATH}' not found.")
        print(f"Please ensure the template is in the working directory.")
        return None

    try:
        wb = load_workbook(TEMPLATE_PATH)
        ws = wb["Template"]
    except KeyError:
        print(f"Error: Sheet 'Template' not found in {TEMPLATE_PATH}.")
        return None
    except Exception as e:
        print(f"Error loading workbook: {e}")
        return None

    start_row = 6
    while ws.cell(row=start_row, column=3).value is not None:
        start_row += 1

    filled_count = 0
    for idx, prod_name in enumerate(selected_products):
        if prod_name not in products_db_to_use:
            print(f"Warning: Product '{prod_name}' not found in database. Skipping.")
            continue

        prod = products_db_to_use[prod_name]
        row = start_row + filled_count

        # Validate and fill main_image
        main_image = prod.get("main_image", "")
        if not main_image or main_image == "N/A":
            print(f"Warning: No valid image URL for '{prod_name}'. Using placeholder or skipping.")
            main_image = ""  # Let TikTok template handle missing images

        ws.cell(row=row, column=1, value=prod.get("category", "Uncategorized"))
        ws.cell(row=row, column=2, value=BRAND_NAME)
        ws.cell(row=row, column=3, value=prod_name)
        ws.cell(row=row, column=4, value=prod.get("description", ""))
        ws.cell(row=row, column=5, value=main_image)
        ws.cell(row=row, column=14, value="Color")
        ws.cell(row=row, column=15, value=prod.get("color", "N/A"))
        ws.cell(row=row, column=19, value=prod.get("weight", 300))

        dims = prod.get("dimensions", (25, 20, 10))
        ws.cell(row=row, column=20, value=dims[0])
        ws.cell(row=row, column=21, value=dims[1])
        ws.cell(row=row, column=22, value=dims[2])

        price = prod.get("price", 89.9)
        ws.cell(row=row, column=24, value=price)
        ws.cell(row=row, column=26, value=100)  # Stock quantity

        product_url = prod.get("product_url", "")
        sku = generate_stable_sku(prod_name, product_url)
        ws.cell(row=row, column=27, value=sku)
        ws.cell(row=row, column=28, value="Yes")

        if prod_name not in uploaded:
            uploaded.append(prod_name)

        filled_count += 1

    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    output_path = f"{OUTPUT_DIR}/TikTok_MUET_Batch_{timestamp}.xlsx"
    
    try:
        wb.save(output_path)
        save_uploaded()
        print(f"✓ Successfully filled template with {filled_count} products.")
        return output_path
    except Exception as e:
        print(f"Error saving workbook: {e}")
        return None

if __name__ == "__main__":
    # Ensure products_db is defined locally if not in global scope
    current_db = globals().get('products_db', {})

    if not current_db:
        print("Note: products_db not found in global scope.")
        print("Please run the 'Integrating Scraped Data' cell first to populate products_db.")
    else:
        available = [p for p in current_db.keys() if p not in uploaded]
        if not available:
            print(f"✓ All {len(current_db)} products have been processed!")
            print(f"To reprocess, either clear {TRACK_FILE} or set CLEAR_UPLOAD_TRACKER=True in integration cell.")
        else:
            selected = available[:BATCH_SIZE]
            print(f"Processing batch of {len(selected)} products ({len(uploaded)} already uploaded)...")
            output = fill_template(selected, current_db)
            if output:
                print(f"\n✓ Success! Generated: {output}")

## Web Scraping for MUET Products (Robust Shopify Endpoint)

This section scrapes product information from the Bauhaus Hong Kong website using the Shopify JSON API endpoint.

**Legal/Ethical Notes:**
- Uses the public Shopify JSON API (common for e-commerce sites).
- Respects `robots.txt` and rate-limits requests (1 second delay between pages).
- Verify terms of service before using scraped data commercially.
- Monitor for changes to the endpoint structure.

In [ ]:
import requests
import pandas as pd
import time
from urllib.parse import urljoin

def check_robots_txt(domain):
    """
    Check robots.txt to verify scraping is allowed.
    Returns True if safe to proceed, False otherwise.
    """
    try:
        robots_url = f"{domain}/robots.txt"
        response = requests.get(robots_url, timeout=5)
        if response.status_code == 200:
            content = response.text.lower()
            # Simple check: warn if User-agent: * is followed by Disallow: /
            if "user-agent: *" in content and "disallow: /" in content:
                print(f"⚠ Warning: {domain}/robots.txt disallows scraping. Proceed with caution.")
                return False
            else:
                print(f"✓ {domain}/robots.txt allows scraping.")
                return True
        else:
            print(f"Note: Could not fetch {robots_url} (status {response.status_code}). Proceeding with caution.")
            return True
    except Exception as e:
        print(f"Note: Could not check robots.txt: {e}. Proceeding with caution.")
        return True

def scrape_bauhaus_muet():
    """
    Scrape MUET products from Bauhaus HK using the Shopify JSON endpoint.
    Validates pagination, handles missing data, and normalizes currency.
    """
    base_url = "https://www.bauhaus.com.hk/en/collections/muet/products.json"
    domain = "https://www.bauhaus.com.hk"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "application/json"
    }

    # Check robots.txt compliance
    if not check_robots_txt(domain):
        print("Scraping may violate robots.txt. Proceeding at your own discretion.")
        response = input("Continue? (yes/no): ").strip().lower()
        if response != "yes":
            return None

    extracted_data = []
    page = 1
    limit = 250
    max_pages = 100  # Safety limit to prevent infinite loops

    print(f"\nInitiating scrape of the MUET collection from {base_url}...")
    print(f"Limit per page: {limit}, Max pages: {max_pages}\n")

    while page <= max_pages:
        url = f"{base_url}?limit={limit}&page={page}"
        print(f"Requesting page {page}...", end="")

        try:
            response = requests.get(url, headers=headers, timeout=10)
            
            if response.status_code == 404:
                # Endpoint changed or pagination ended
                print(f" [404] Page not found. Pagination may have ended.")
                break
            elif response.status_code != 200:
                print(f" [Error {response.status_code}] Stopping.")
                break

            data = response.json()
            products = data.get("products", [])
            
            if not products:
                print(f" [Empty] No products returned. Pagination ended.")
                break

            print(f" [{len(products)} products]")

            for product in products:
                variants = product.get("variants", [])
                images = product.get("images", [])

                # Robust price extraction
                price = "N/A"
                if variants and len(variants) > 0:
                    price_val = variants[0].get("price")
                    if price_val:
                        try:
                            price = float(price_val)
                        except (ValueError, TypeError):
                            price = "N/A"

                # Robust image extraction
                image_url = "N/A"
                if images and len(images) > 0:
                    img_src = images[0].get("src")
                    if img_src:
                        image_url = img_src

                extracted_data.append({
                    "Product Name": product.get("title", "Unknown"),
                    "Price": price,
                    "Image URL": image_url,
                    "Product URL": f"https://www.bauhaus.com.hk/en/products/{product.get('handle', '')}"
                })

            time.sleep(1)  # Rate limiting: 1 second between requests
            page += 1
            
        except requests.exceptions.Timeout:
            print(f" [Timeout] Request took too long. Retrying...")
            time.sleep(2)
            continue
        except requests.exceptions.RequestException as e:
            print(f" [Connection Error] {e}")
            break
        except ValueError as e:
            print(f" [JSON Parse Error] {e}")
            break

    if extracted_data:
        df = pd.DataFrame(extracted_data)
        csv_path = "bauhaus_muet_products.csv"
        df.to_csv(csv_path, index=False)
        print(f"\n✓ Scraping complete. Extracted {len(df)} products saved to '{csv_path}'.")
        
        # Show summary
        na_count = (df["Image URL"] == "N/A").sum()
        if na_count > 0:
            print(f"⚠ {na_count} products have missing image URLs.")
        
        display(df.head())
        return df
    else:
        print("✗ No data retrieved.")
        return None

if __name__ == "__main__":
    df_scraped_products = scrape_bauhaus_muet()

## Integrating Scraped Data into `products_db`

This section processes the `df_scraped_products` DataFrame and transforms it into the format required for the template filler.

**Important:** By default, this preserves previously uploaded products (no duplicates). To reset tracking and reprocess all products, set `CLEAR_UPLOAD_TRACKER = True` in the configuration below.

In [ ]:
import pandas as pd
import os
import re

# ========== INTEGRATION CONFIG ==========
CLEAR_UPLOAD_TRACKER = False  # Set to True to reset and reprocess ALL products

# Configuration Defaults (Ensure these match your main config cell)
DEFAULT_PRICE = 89.90
DEFAULT_WEIGHT = 300
DEFAULT_LENGTH = 25
DEFAULT_WIDTH = 20
DEFAULT_HEIGHT = 10

def parse_price(price_raw):
    """
    Robustly parse price from various formats.
    Handles HK$ currency symbols, commas, and missing values.
    Returns float or DEFAULT_PRICE on error.
    """
    try:
        if pd.isna(price_raw) or price_raw == "N/A" or price_raw == "":
            return DEFAULT_PRICE
        
        # Convert to string and clean
        price_str = str(price_raw).strip()
        
        # Remove common currency symbols and separators
        price_str = re.sub(r'[HK$,\s]+', '', price_str)
        
        # Try to parse as float
        if price_str:
            return float(price_str)
        else:
            return DEFAULT_PRICE
    except (ValueError, TypeError):
        return DEFAULT_PRICE

try:
    # Optionally clear upload tracker
    if CLEAR_UPLOAD_TRACKER:
        global uploaded
        uploaded = []
        print("✓ Upload tracker cleared. All products will be reprocessed.")

    # Load the scraped data from the CSV generated in the previous step
    csv_path = 'bauhaus_muet_products.csv'
    if not os.path.exists(csv_path):
        print(f"Error: {csv_path} not found.")
        print("Please run the 'Web Scraping' cell first.")
    else:
        df_scraped_products = pd.read_csv(csv_path)
        print(f"Loaded {len(df_scraped_products)} products from {csv_path}.")

        if 'df_scraped_products' in locals() and not df_scraped_products.empty:
            new_products_db = {}
            errors_count = 0

            for index, row in df_scraped_products.iterrows():
                product_name = str(row.get('Product Name', 'Unknown Product')).strip()

                if not product_name or product_name == "Unknown Product":
                    print(f"Warning: Skipping row {index} with invalid product name.")
                    errors_count += 1
                    continue

                # Robust price parsing
                price = parse_price(row.get('Price'))

                # Robust image URL handling
                image_url = str(row.get('Image URL', '')).strip()
                if image_url == "N/A" or not image_url:
                    image_url = ""  # Empty string is safer than N/A for URLs
                    # Uncomment to warn:
                    # print(f"Note: {product_name} has no image URL.")

                product_url = str(row.get('Product URL', '')).strip()

                # Map to the format required by the TikTok template filler
                new_products_db[product_name] = {
                    "category": "Uncategorized",
                    "description": f"Official MUET product: {product_name}. Imported from Bauhaus Hong Kong.",
                    "leather_type": "Synthetic / Unknown",
                    "main_image": image_url,
                    "images": [image_url] if image_url else [],
                    "color": "Multi",
                    "size": "One Size",
                    "price": price,
                    "weight": DEFAULT_WEIGHT,
                    "dimensions": (DEFAULT_LENGTH, DEFAULT_WIDTH, DEFAULT_HEIGHT),
                    "product_url": product_url
                }

            # Update the global products_db variable
            products_db = new_products_db

            print(f"\n✓ Successfully processed {len(new_products_db)} products into products_db.")
            if errors_count > 0:
                print(f"⚠ Skipped {errors_count} rows due to errors.")
            print(f"✓ Ready to generate upload file. ({len(uploaded)} products already uploaded.)")
            display(pd.DataFrame.from_dict(products_db, orient='index').head())
        else:
            print("Error: df_scraped_products is empty or not found.")

except Exception as e:
    print(f"An error occurred during integration: {e}")
    import traceback
    traceback.print_exc()